We investigate:
1. **Baseline** — Zero-shot prompting with a small instruction-tuned LLM
2. **Prompt engineering** — Zero-shot vs. few-shot vs. chain-of-thought
3. **RAG** — Retrieval-Augmented Generation using Wikipedia
4. **Tool use** — Calculator/SymPy for the Maths category
5. **Multi-model ensemble** — Majority vote across models
6. **Evaluation** — Accuracy by category, level, and model; response time analysis


## 0. Setup — Mount Drive & Install Dependencies

In [ ]:
# Connect to Google Drive
from google.colab import drive
drive.mount('/content/gdrive/')

In [ ]:
# The package path
import sys, os

package_parent_dir = '/content/gdrive/MyDrive/NLP_assignment'
if package_parent_dir not in sys.path:
    sys.path.append(package_parent_dir)

print("Path added, it has been:", package_parent_dir)

In [ ]:
# Install required package
!pip install -q transformers accelerate bitsandbytes sentencepiece sympy wikipedia-api openai-whisper
!pip install -q torch --index-url https://download.pytorch.org/whl/cu118


In [ ]:
# Import the client
from millionaire_client import MillionaireClient, AuthenticationError
import time, json, re, random
from datetime import datetime
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

print("Imports complete")


## 1. Login & Explore the Game

In [ ]:
# Credentials(store in Colab Secrets as 'poli-millionaire')
from google.colab import userdata

API_URL = (__import__("os").getenv("POLI_MILLIONAIRE_API_URL") or input("PoliMillionaire API URL: ").strip())
USERNAME = (__import__("os").getenv("POLI_MILLIONAIRE_USERNAME") or input("PoliMillionaire username: ").strip())
PASSWORD = (__import__("os").getenv("POLI_MILLIONAIRE_PASSWORD") or __import__("getpass").getpass("PoliMillionaire password: ").strip())
client = MillionaireClient(API_URL)
try:
    user = client.login(USERNAME, PASSWORD)
    print(f"Welcome, {user.username}! (Role: {user.role})")
except AuthenticationError as e:
    print(f"Login failed, it has: {e}")

In [ ]:
# List all competitions
print("=== Available Competitions ===")
competitions = client.competitions.list_all()
for comp in competitions:
    print(f"  [{comp.id}] {comp.name} — {comp.max_levels} questions | {comp.description}")

## 2. Baseline Model — Zero-Shot with QWEN



In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "Qwen/Qwen2.5-7B-Instruct"

# 4-bit configuration keeps it safe for free Colab hardware
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Model successfully loaded onto the T4 GPU!")

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
import re
import time
import torch

#zero-shot prompt
def build_zero_shot_prompt(question_text, options):
    # Format options as A) B) C) D)
    opts = "\n".join([f"{chr(65+i)}) {o.text}" for i, o in enumerate(options)])
    return (
        f"Answer the following multiple choice question. "
        f"Reply with only the letter A, B, C, or D.\n\n"
        f"Question: {question_text}\n{opts}\n\nAnswer:"
    )

def extract_letter(text):
    # 1. Clean the text
    text = text.strip()

    # 2. Look for explicit patterns like "Answer: B" or "Final Answer: [B]" near the end
    match = re.search(r"(?:FINAL ANSWER|ANSWER|OPTION):\s*([A-D])", text.upper())
    if match:
        return match.group(1)

    # 3. Fallback: Find all isolated capital letters A, B, C, D and take the LAST one
    letters = re.findall(r"\b([A-D])\b", text.upper())
    if letters:
        return letters[-1]  # Takes the final decision made by the model

    return "A"  # Default fallback guess

def answer_with_model(question, prompt_fn=build_zero_shot_prompt, max_new_tokens=128):
    # Time the response
    t0 = time.time()

    # 1. Generate your standard question string
    raw_prompt = prompt_fn(question.text, question.options)

    # 2. Format it into the model's chat structure
    messages = [{"role": "user", "content": raw_prompt}]
    formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # 3. Tokenize the formatted prompt
    inputs = tokenizer(formatted_prompt, return_tensors="pt", truncation=True, max_length=512).to(device)

    # 4. Generate the answer
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.2,
            do_sample=False,
            repetition_penalty=1.2,
            pad_token_id=tokenizer.eos_token_id
        )

    # 5. CRITICAL FIX: Only extract tokens generated AFTER the prompt sequence length
    prompt_length = inputs.input_ids.shape[1]
    new_generated_tokens = outputs[0][prompt_length:]

    # Decode ONLY the new answer text
    response = tokenizer.decode(new_generated_tokens, skip_special_tokens=True)

    # Extract the choice letter from the clean text response
    letter = extract_letter(response)
    elapsed = time.time() - t0

    # Map letter to option ID (with safety lower boundary check)
    idx = ord(letter) - ord("A")
    idx = min(max(0, idx), len(question.options) - 1)

    return question.options[idx].id, letter, elapsed, response

print("Answer function defined.")

### Speech Mode — Whisper Medium With Cleanup

This block lets the same game loop run with `mode="speech"`. It uses Whisper `medium`, retries empty option clips, and removes common TTS/Whisper noise before the answer model sees the text.


In [ ]:
# Speech-mode transcription helpers: Whisper turbo + noise cleanup + empty-option retries
import gc
import traceback
import re
import time
from pathlib import Path

WHISPER_MODEL_SIZE = "turbo"
WHISPER_FALLBACK_MODEL_SIZES = ["large-v3", "medium", "small", "base"]
WHISPER_DEVICE = "cuda"
WHISPER_LANGUAGE = "en"
WHISPER_FP16 = True
WHISPER_RETRY_EMPTY_OPTIONS = True
WHISPER_MIN_TRANSCRIPT_CHARS = 2
SAVE_SPEECH_AUDIO = True
DISPLAY_SPEECH_AUDIO = False
SPEECH_AUDIO_DIR = "/content/gdrive/MyDrive/Colab Notebooks/NLP_Test/speech_game_audio"

_WHISPER_MODEL_CACHE = {}
ACTIVE_WHISPER_MODEL_SIZE = None
ACTIVE_WHISPER_DEVICE = None


def print_cuda_memory(label):
    try:
        import torch
        if not torch.cuda.is_available():
            print(f"{label}: CUDA not available")
            return
        free_bytes, total_bytes = torch.cuda.mem_get_info()
        gb = 1024 ** 3
        print(
            f"{label}: "
            f"free={free_bytes / gb:.2f}GB | "
            f"total={total_bytes / gb:.2f}GB | "
            f"allocated={torch.cuda.memory_allocated() / gb:.2f}GB | "
            f"reserved={torch.cuda.memory_reserved() / gb:.2f}GB"
        )
    except Exception as exc:
        print(f"{label}: CUDA memory check unavailable ({exc})")


def load_whisper_model_for_speech():
    import torch
    import whisper

    requested_device = WHISPER_DEVICE
    device_name = requested_device if requested_device == "cpu" or torch.cuda.is_available() else "cpu"
    candidate_sizes = [WHISPER_MODEL_SIZE]
    if device_name == "cuda":
        for fallback_size in WHISPER_FALLBACK_MODEL_SIZES:
            if fallback_size not in candidate_sizes:
                candidate_sizes.append(fallback_size)

    last_oom = None
    for model_size in candidate_sizes:
        cache_key = (model_size, device_name)
        if cache_key in _WHISPER_MODEL_CACHE:
            globals()["ACTIVE_WHISPER_MODEL_SIZE"] = model_size
            globals()["ACTIVE_WHISPER_DEVICE"] = device_name
            return _WHISPER_MODEL_CACHE[cache_key], device_name

        try:
            if device_name == "cuda":
                torch.cuda.empty_cache()
            print_cuda_memory(f"Before Whisper {model_size} load")
            print(f"Loading Whisper {model_size!r} on {device_name}...")
            started_at = time.time()
            _WHISPER_MODEL_CACHE[cache_key] = whisper.load_model(model_size, device=device_name)
            globals()["ACTIVE_WHISPER_MODEL_SIZE"] = model_size
            globals()["ACTIVE_WHISPER_DEVICE"] = device_name
            print(f"Whisper ready in {time.time() - started_at:.1f}s")
            print_cuda_memory(f"After Whisper {model_size} load")
            return _WHISPER_MODEL_CACHE[cache_key], device_name
        except RuntimeError as exc:
            message = str(exc).lower()
            if device_name == "cuda" and ("out of memory" in message or "cuda" in message):
                last_oom = RuntimeError(str(exc))
                traceback.clear_frames(exc.__traceback__)
                print(f"Whisper {model_size!r} did not fit on CUDA; trying fallback.")
                _WHISPER_MODEL_CACHE.pop(cache_key, None)
                del exc
                gc.collect()
                torch.cuda.empty_cache()
                try:
                    torch.cuda.ipc_collect()
                except Exception:
                    pass
                print_cuda_memory(f"After Whisper {model_size} OOM cleanup")
                continue
            raise

    if device_name == "cuda":
        print("Whisper did not fit on CUDA; retrying large-v3 on CPU.")
        device_name = "cpu"
        cache_key = (WHISPER_MODEL_SIZE, device_name)
        if cache_key not in _WHISPER_MODEL_CACHE:
            _WHISPER_MODEL_CACHE[cache_key] = whisper.load_model(WHISPER_MODEL_SIZE, device=device_name)
        globals()["ACTIVE_WHISPER_MODEL_SIZE"] = WHISPER_MODEL_SIZE
        globals()["ACTIVE_WHISPER_DEVICE"] = device_name
        return _WHISPER_MODEL_CACHE[cache_key], device_name

    raise RuntimeError("Could not load Whisper model") from last_oom


def clean_whisper_text(text):
    text = re.sub(r"\s+", " ", str(text or "")).strip()
    return text.strip(' \"')


def strip_speech_noise(text):
    text = clean_whisper_text(text)
    if not text:
        return ""

    hallucination_phrases = [
        r"\bthanks? for watching[.!?]*",
        r"\bthank you for watching[.!?]*",
        r"\bbut you all too much for me to download[.!?]*",
        r"\byou all too much for me to download[.!?]*",
    ]
    for pattern in hallucination_phrases:
        text = re.sub(pattern, " ", text, flags=re.IGNORECASE)

    laughter_or_filler = (
        r"(?:\b(?:a?ha(?:ha)+|ha|he(?:he)+h?|ehe(?:he)+h?|ah+|eh+|uh+|um+|ahem|pfft+)\b"
        r"[\s,.;:!?-]*)+"
    )
    previous = None
    while previous != text:
        previous = text
        text = re.sub(laughter_or_filler, " ", text, flags=re.IGNORECASE)

    text = re.sub(r"\s+([,.;:!?])", r"\1", text)
    text = re.sub(r"(?:^|\s)[,.;:!?-]+(?=\s|$)", " ", text)
    text = re.sub(r"\s+", " ", text).strip(" ,.;:!?-")
    return text


def clean_question_transcript(text):
    text = strip_speech_noise(text)
    text = re.sub(r"^(?:oh|uh|um|ahem)[,!.?\s]+", "", text, flags=re.IGNORECASE).strip()
    return clean_whisper_text(text)


def clean_option_transcript(text, letter):
    text = strip_speech_noise(text)
    patterns = [
        rf"^Option\s*{letter}\s*[\.:,\)]?\s*",
        r"^Option\s*[A-D]\s*[\.:,\)]?\s*",
        rf"^{letter}\s*[\.:\)]\s*",
    ]
    for pattern in patterns:
        text = re.sub(pattern, "", text, flags=re.IGNORECASE).strip()
    return strip_speech_noise(text)


def transcript_has_content(text):
    return len(re.sub(r"[^A-Za-z0-9]", "", text or "")) >= WHISPER_MIN_TRANSCRIPT_CHARS


def maybe_display_audio(audio_bytes):
    if not DISPLAY_SPEECH_AUDIO:
        return
    try:
        from IPython.display import Audio, display
        display(Audio(audio_bytes))
    except Exception as exc:
        print(f"Could not display audio inline: {exc}")


def save_speech_audio(audio_bytes, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "wb") as handle:
        handle.write(audio_bytes)
    return path


def transcribe_audio_file(audio_path, initial_prompt=None, is_option=False, letter=None):
    whisper_model, whisper_device = load_whisper_model_for_speech()
    fp16 = bool(WHISPER_FP16 and whisper_device == "cuda")
    attempts = [
        {
            "initial_prompt": initial_prompt,
            "temperature": 0.0,
            "no_speech_threshold": 0.95,
            "logprob_threshold": -1.5,
            "compression_ratio_threshold": 2.8,
        },
        {
            "initial_prompt": None,
            "temperature": 0.0,
            "no_speech_threshold": 1.0,
            "logprob_threshold": -2.0,
            "compression_ratio_threshold": 3.5,
            "suppress_blank": False,
        },
        {
            "initial_prompt": None,
            "temperature": 0.2,
            "no_speech_threshold": 1.0,
            "logprob_threshold": -2.0,
            "compression_ratio_threshold": 3.5,
            "suppress_blank": False,
        },
    ]
    if not (is_option and WHISPER_RETRY_EMPTY_OPTIONS):
        attempts = attempts[:1]

    best_text = ""
    best_raw_text = ""
    for attempt_index, attempt in enumerate(attempts, start=1):
        kwargs = {
            "language": WHISPER_LANGUAGE,
            "task": "transcribe",
            "fp16": fp16,
            "condition_on_previous_text": False,
            **attempt,
        }
        if float(kwargs.get("temperature", 0.0)) == 0.0:
            kwargs["beam_size"] = 5
        else:
            kwargs["best_of"] = 5
        prompt = kwargs.pop("initial_prompt", None)
        if prompt:
            kwargs["initial_prompt"] = prompt

        result = whisper_model.transcribe(str(audio_path), **kwargs)
        raw_text = clean_whisper_text(result.get("text", ""))
        cleaned_text = clean_option_transcript(raw_text, letter or "") if is_option else clean_question_transcript(raw_text)

        if cleaned_text and (not best_text or len(cleaned_text) > len(best_text)):
            best_text = cleaned_text
            best_raw_text = raw_text

        if transcript_has_content(cleaned_text):
            if raw_text != cleaned_text:
                print(f"Cleaned transcript noise: {raw_text!r} -> {cleaned_text!r}")
            return cleaned_text

        if is_option and attempt_index < len(attempts):
            print(f"Empty/low-content option transcript from {audio_path.name}; retrying Whisper pass {attempt_index + 1}...")

    if best_raw_text and best_raw_text != best_text:
        print(f"Cleaned transcript noise: {best_raw_text!r} -> {best_text!r}")
    return best_text


def transcribe_speech_question(game):
    question = game.current_question
    if question is None:
        return None, {"error": "No active question returned by server."}

    audio_dir = Path(SPEECH_AUDIO_DIR)
    level = game.current_level
    session_id = game.session_id
    transcript = {
        "mode": "speech",
        "session_id": session_id,
        "level": level,
        "audio_files": {},
        "question": None,
        "options": [],
        "whisper_model_size": globals().get("ACTIVE_WHISPER_MODEL_SIZE") or WHISPER_MODEL_SIZE,
        "whisper_device": globals().get("ACTIVE_WHISPER_DEVICE") or WHISPER_DEVICE,
    }
    started_at = time.time()

    def option_letter(index):
        letters = globals().get("LETTERS", "ABCD")
        return letters[index] if index < len(letters) else chr(65 + index)

    def audio_path_for(kind, letter=None):
        if kind == "question":
            filename = f"session_{session_id}_level_{level}_question.wav"
        else:
            filename = f"session_{session_id}_level_{level}_option_{letter}.wav"
        if SAVE_SPEECH_AUDIO:
            return audio_dir / filename
        return Path("/tmp") / filename

    # Fetch every server audio clip first. Do not run Whisper between these calls:
    # the API can expire the question while option audio is still being delivered.
    print("Fetching question audio...")
    question_audio = game.fetch_audio_question()
    question_path = audio_path_for("question")
    save_speech_audio(question_audio, question_path)
    transcript["audio_files"]["question"] = str(question_path)
    maybe_display_audio(question_audio)

    option_paths = []
    option_count = len(question.options) or 4
    for index in range(option_count):
        letter = option_letter(index)
        print(f"Fetching option {letter} audio...")
        option_audio = game.fetch_audio_option_next()
        option_path = audio_path_for("option", letter)
        save_speech_audio(option_audio, option_path)
        transcript["audio_files"][letter] = str(option_path)
        option_paths.append((letter, option_path))
        maybe_display_audio(option_audio)

    # The timer is active after the option audio sequence. Refresh immediately so
    # later time checks use the server deadline, then transcribe local WAV files.
    try:
        game.refresh_state()
        refreshed_question = game.current_question
        if refreshed_question is not None:
            question = refreshed_question
    except Exception as exc:
        print(f"Could not refresh game state after speech audio delivery: {exc}")

    print("Transcribing question audio...")
    question_text = transcribe_audio_file(question_path, initial_prompt="A multiple choice trivia question.")
    question.text = question_text
    transcript["question"] = question_text
    print("Question transcript:", question_text)

    option_texts = []
    for index, (letter, option_path) in enumerate(option_paths):
        option_text = transcribe_audio_file(option_path, initial_prompt=None, is_option=True, letter=letter)
        option_texts.append(option_text)
        transcript["options"].append({"letter": letter, "audio_file": str(option_path), "text": option_text})
        print(f"Option {letter} transcript: {option_text}")

    for index, option in enumerate(question.options):
        if index < len(option_texts):
            option.text = option_texts[index]

    transcript["transcription_seconds"] = time.time() - started_at
    try:
        transcript["seconds_left_after_audio"] = seconds_available(game)
    except NameError:
        transcript["seconds_left_after_audio"] = game.time_remaining
    transcript["whisper_model_size"] = globals().get("ACTIVE_WHISPER_MODEL_SIZE") or WHISPER_MODEL_SIZE
    transcript["whisper_device"] = globals().get("ACTIVE_WHISPER_DEVICE") or WHISPER_DEVICE
    return question, transcript

print("Speech helpers ready. Use play_full_game(..., mode='speech') to play with audio.")


## 3. Game Loop & Evaluation Harness

In [ ]:
# The game loop
def play_full_game(competition_id, answer_fn, label="Model", mode="text"):
    """
    Play a complete game and return results log.
    answer_fn: callable(question) -> (option_id, letter, elapsed, raw_response)
    mode: "text" or "speech". Speech mode fetches audio and transcribes it with Whisper.
    """
    if mode not in {"text", "speech"}:
        raise ValueError('mode must be either "text" or "speech"')

    if mode == "speech":
        load_whisper_model_for_speech()

    game = client.game.start(competition_id=competition_id, mode=mode)
    print(f"\n=== Game Started: {label} | Competition {competition_id} | Mode {game.mode} | Session {game.session_id} ===")

    log = []

    while game.in_progress:
        speech_transcription = None
        if game.mode == "speech":
            q, speech_transcription = transcribe_speech_question(game)
        else:
            q = game.current_question

        if not q:
            print("No question available, there is. Ending, the game is.")
            break

        time_left = game.time_remaining
        time_left_text = f"{time_left:.1f}s" if time_left is not None else "n/a"
        current_level = game.current_level
        print(f"\n--- Level {current_level} | Time left: {time_left_text} ---")
        if speech_transcription:
            print(
                "Speech transcription:",
                f"{speech_transcription.get('transcription_seconds', 0.0):.1f}s",
                f"| {speech_transcription.get('seconds_left_after_audio', time_left) or 0:.1f}s left after audio",
                f"| Whisper {speech_transcription.get('whisper_model_size')} on {speech_transcription.get('whisper_device')}",
            )
        print(f"Q: {q.text}")
        for opt in q.options:
            print(f"   [{opt.id}] {opt.text}")

        # Get answer from the model
        try:
            option_id, letter, elapsed, raw = answer_fn(q)
        except Exception as e:
            print(f"Model error, there is: {e}. Random answer, choosing we are.")
            option_id = random.choice(q.options).id
            letter, elapsed, raw = "?", 0.0, str(e)

        print(f"   -> Chose: {letter} (in {elapsed:.2f}s)")

        # Submit answer
        result = game.answer(option_id)

        entry = {
            "level": current_level,
            "question": q.text,
            "options": [{"id": opt.id, "text": opt.text} for opt in q.options],
            "correct": result.correct,
            "timed_out": result.timed_out,
            "elapsed": elapsed,
            "chosen_letter": letter,
            "earned": result.earned_amount,
            "model_raw": raw,
            "mode": game.mode,
            "speech_transcription": speech_transcription,
        }
        log.append(entry)

        if result.timed_out:
            print("   TIMED OUT!")
            break
        elif result.correct:
            print(f"   CORRECT! Earned: ${result.earned_amount:,.0f}")
            if result.game_over:
                print("   GAME COMPLETE!")
                break
        else:
            print(f"   WRONG! Final earnings: ${result.earned_amount:,.0f}")
            break

    print(f"\n=== Game Over | Reached Level: {game.current_level} | Earnings: ${game.earned_amount:,.0f} ===")
    return log, game.current_level, game.earned_amount


## 4. Run Baseline — Zero-Shot Flan-T5

In [ ]:
# Competition IDs: 0=Entertainment, 1=Ancient History & Politics, 2=Science & Nature, 3=Maths, 4=Philosophy & Psychology, 5=News
COMP_ID = 0

baseline_log, baseline_level, baseline_earned = play_full_game(
    competition_id=COMP_ID,
    answer_fn=lambda q: answer_with_model(q, build_zero_shot_prompt),
    label="QWEN Zero-Shot",
    mode="speech"
)

## 5. Prompt Engineering — Few-Shot & Chain-of-Thought

We test whether giving the model examples (few-shot) or asking it to reason before answering (chain-of-thought) improves accuracy.


In [ ]:
# Entertainment specialized few-shot data
FEW_SHOT_EXAMPLES = [
    {
        "question": "Which movie won the Academy Award for Best Picture in 2020?",
        "options": ["A) 1917", "B) Parasite", "C) Joker", "D) Once Upon a Time in Hollywood"],
        "answer": "B"
    },
    {
        "question": "Who is widely recognized as the 'King of Pop'?",
        "options": ["A) Elvis Presley", "B) Prince", "C) Michael Jackson", "D) Madonna"],
        "answer": "C"
    }
]

def build_few_shot_prompt(question_text, options):
    shots = ""
    for ex in FEW_SHOT_EXAMPLES:
        shots += f"Question: {ex['question']}\n" + "\n".join(ex['options']) + f"\nAnswer: {ex['answer']}\n\n"
    opts = "\n".join([f"{chr(65+i)}) {o.text}" for i, o in enumerate(options)])
    return (
        f"Answer multiple choice questions with only a single letter A, B, C, or D.\n\n"
        f"{shots}"
        f"Question: {question_text}\n{opts}\nAnswer:"
    )

def build_cot_prompt(question_text, options):
    opts = "\n".join([f"{chr(65+i)}) {o.text}" for i, o in enumerate(options)])
    return (
        f"Answer the following question. Think briefly, then give your final answer as a single letter.\n\n"
        f"Question: {question_text}\n{opts}\n\n"
        f"Reasoning: Let me think step by step.\nFinal Answer:"
    )

print("Prompt variants successfully adjusted for Entertainment trivia parsing.")

In [ ]:
# Compare prompts offline on a sample — NOT via live game (save API calls)
# Manually define a test question to compare prompt styles
sample_text = "Which planet is known as the Red Planet?"

class FakeOption:
    def __init__(self, id_, text):
        self.id = id_
        self.text = text

sample_opts = [FakeOption(1,"Mars"), FakeOption(2,"Venus"), FakeOption(3,"Jupiter"), FakeOption(4,"Saturn")]

class FakeQ:
    def __init__(self):
        self.text = sample_text
        self.options = sample_opts

fq = FakeQ()

print("=== Zero-Shot ===")
print(build_zero_shot_prompt(fq.text, fq.options))
print("\n=== Few-Shot ===")
print(build_few_shot_prompt(fq.text, fq.options))
print("\n=== Chain-of-Thought ===")
print(build_cot_prompt(fq.text, fq.options))

In [ ]:
# Run few-shot game — compare with baseline result above
fewshot_log, fewshot_level, fewshot_earned = play_full_game(
    competition_id=COMP_ID,
    answer_fn=lambda q: answer_with_model(q, build_few_shot_prompt),
    label="Few-Shot",
    mode="speech"
)

## 6. RAG — Retrieval-Augmented Generation with Wikipedia

For factual questions (History, Science, Entertainment), we search Wikipedia and inject the retrieved passage as context into the prompt. This gives the model access to external knowledge without violating the no-LLM-API rule.


In [ ]:
import requests
import re
import time

def search_wikipedia_deep(query, max_chars=1200):
    """
    Deeper Wikipedia extract grabber to catch tracklists, cast lists,
    and detailed table indexes missing from short summaries.
    """
    clean = re.sub(r'[^\w\s]', '', query)[:60].strip()

    # Phase 1: Search API to get the correct matching title
    search_url = "https://en.wikipedia.org/w/api.php"
    search_params = {
        "action": "query", "list": "search",
        "srsearch": clean, "format": "json", "srlimit": 1
    }
    try:
        r = requests.get(search_url, params=search_params, timeout=5).json()
        results = r.get("query", {}).get("search", [])
        if not results:
            return ""
        title = results[0]["title"]

        # Phase 2: Request full un-summarized text section extract
        content_params = {
            "action": "query", "prop": "extracts",
            "explaintext": 1, "titles": title, "format": "json", "exintro": 0
        }
        resp = requests.get(search_url, params=content_params, timeout=5).json()
        pages = resp["query"]["pages"]
        page_id = list(pages.keys())[0]

        extract = pages[page_id].get("extract", "")
        return extract[:max_chars]
    except Exception:
        return ""

def extract_entertainment_query(question_text, options):
    """
    Concatenates target choices to the question search query
    so negative constraint tracking works properly.
    """
    # Isolate key elements like text in quotes (e.g. "Born to Die")
    quoted_terms = re.findall(r'"([^"]*)"', question_text)
    options_string = " ".join([o.text for o in options])

    # Strip common filler stop phrases
    clean_q = re.sub(r'(Which of these|is not|featured on|the standard version of|the following|correct answer)', '', question_text, flags=re.IGNORECASE)

    if quoted_terms:
        return f'"{quoted_terms[0]}" {options_string}'[:90]
    return f"{clean_q.strip()} {options_string}"[:90]

def build_entertainment_rag_prompt(question_text, options, context=""):
    """
    Advanced prompt directing process of elimination for negative properties (NOT, EXCEPT).
    """
    opts = "\n".join([f"{chr(65+i)}) {o.text}" for i, o in enumerate(options)])
    ctx_block = f"Background Context Information:\n{context}\n\n" if context else ""

    return (
        f"{ctx_block}"
        f"Task: Solve this multiple choice entertainment trivia question based ONLY on the text above.\n"
        f"CRITICAL RULES:\n"
        f"1. If the question contains words like 'NOT', 'FALSE', or 'EXCEPT', use process of elimination. Eliminate any options explicitly verified by the context text and pick the one outlier that remains.\n"
        f"2. Output strictly a single capital letter matching the answer (A, B, C, or D).\n\n"
        f"Question: {question_text}\n"
        f"Options:\n{opts}\n\n"
        f"Final Answer (Letter Only):"
    )

def answer_with_rag(question):
    # Pass options array to feed query synthesis loop
    query = extract_entertainment_query(question.text, question.options)
    context = search_wikipedia_deep(query)

    if context:
        print(f"   [RAG Active] Retrieved context data frame for search parameters.")
    else:
        print("   [RAG Warning] Flying blind without structural text records.")

    return answer_with_model(
        question,
        lambda q_text, opts: build_entertainment_rag_prompt(q_text, opts, context),
        max_new_tokens=16 # Kept short to prevent model from writing chat justifications
    )

In [ ]:
import requests
import re
import time

def search_wikipedia_deep(query, max_chars=1200):
    """
    Deeper Wikipedia extract grabber to catch tracklists, cast lists,
    and detailed table indexes missing from short summaries.
    """
    clean = re.sub(r'[^\w\s]', '', query)[:60].strip()

    # Phase 1: Search API to get the correct matching title
    search_url = "https://en.wikipedia.org/w/api.php"
    search_params = {
        "action": "query", "list": "search",
        "srsearch": clean, "format": "json", "srlimit": 1
    }
    try:
        r = requests.get(search_url, params=search_params, timeout=5).json()
        results = r.get("query", {}).get("search", [])
        if not results:
            return ""
        title = results[0]["title"]

        # Phase 2: Request full un-summarized text section extract
        content_params = {
            "action": "query", "prop": "extracts",
            "explaintext": 1, "titles": title, "format": "json", "exintro": 0
        }
        resp = requests.get(search_url, params=content_params, timeout=5).json()
        pages = resp["query"]["pages"]
        page_id = list(pages.keys())[0]

        extract = pages[page_id].get("extract", "")
        return extract[:max_chars]
    except Exception:
        return ""
def extract_news_query(question_text, options):
    """
    Optimized for Wikipedia News search: Extracts key entities and dates
    while ignoring option clutter to ensure clean page hits.
    """
    # 1. Heavily prioritize anything inside quotation marks (e.g., specific event titles or treaties)
    quoted_terms = re.findall(r'"([^"]*)"', question_text)
    if quoted_terms:
        return quoted_terms[0][:60]

    # 2. Extract Capitalized Proper Nouns (Entities) and Years
    proper_nouns = re.findall(r'\b[A-Z][a-zA-Z0-9_]+\b', question_text)
    years = re.findall(r'\b\d{4}\b', question_text)

    # Filter out common question words that happen to start with capital letters
    fillers = {"Which", "What", "Who", "When", "Where", "How", "The", "In", "On", "At", "A", "B", "C", "D", "NOT", "FALSE"}
    clean_nouns = [word for word in proper_nouns if word not in fillers]

    # Combine entities and timestamps for a highly focused Wikipedia lookup
    search_terms = clean_nouns + years
    if search_terms:
        return " ".join(search_terms[:4])

    # Fallback to the first 5 meaningful words if no proper nouns are isolated
    words = re.findall(r'\b\w{4,}\b', question_text)
    return " ".join(words[:5])

In [ ]:
def build_news_rag_prompt(question_text, options, context=""):
    """
    Optimized for News and Events: Forces the model to carefully inspect
    historical timelines, official roles, and factual details from the Wikipedia context.
    """
    opts = "\n".join([f"{chr(65+i)}) {o.text}" for i, o in enumerate(options)])
    ctx_block = f"Background Context Information (Wikipedia Extract):\n{context}\n\n" if context else ""

    return (
        f"{ctx_block}"
        f"Task: Solve this current events and news multiple choice question using the background text above.\n"
        f"CRITICAL RULES:\n"
        f"1. Pay close attention to exact dates, years, official political titles, and specific country/geographic actions mentioned in the text.\n"
        f"2. Output strictly a single capital letter matching the correct choice (A, B, C, or D).\n\n"
        f"Question: {question_text}\n"
        f"Options:\n{opts}\n\n"
        f"Final Answer (Letter Only):"
    )

def answer_with_rag(question):
    # Call the new news query extractor instead of entertainment
    query = extract_news_query(question.text, question.options)
    print(f"   [Wikipedia News Search] Query: '{query}'")

    # Keep using your deep lookup engine exactly as it was
    context = search_wikipedia_deep(query)

    if context:
        print(f"   [RAG Active] Retrieved context data frame from Wikipedia.")
    else:
        print("   [RAG Warning] Flying blind without structural text records.")

    return answer_with_model(
        question,
        lambda q_text, opts: build_news_rag_prompt(q_text, opts, context),
        max_new_tokens=16
    )

In [ ]:
# Run RAG game
rag_log, rag_level, rag_earned = play_full_game(
    competition_id=0,
    answer_fn=answer_with_rag,
    label="Model + Wikipedia RAG",
    mode="speech"
)

In [ ]:
# Multi-Source RAG Setup — DuckDuckGo Live Search + Wikipedia Fallback
import requests
import re

def search_duckduckgo(query, max_chars=600):
    """
    Queries DuckDuckGo's free API for an instant abstract summary.
    Perfect for pop-culture entities, famous tracks, actors, and media questions.
    """
    clean_query = query.strip()
    url = f"https://api.duckduckgo.com/?q={requests.utils.quote(clean_query)}&format=json&no_html=1"
    try:
        response = requests.get(url, timeout=4)
        if response.status_code == 200:
            data = response.json()
            # DuckDuckGo provides 'AbstractText' for broad definitions
            abstract = data.get("AbstractText", "")
            if abstract:
                return abstract[:max_chars]

            # Alternative fallback: look inside RelatedTopics list snippets
            related = data.get("RelatedTopics", [])
            if related and "Text" in related[0]:
                return related[0]["Text"][:max_chars]
    except Exception:
        pass
    return ""

def search_wikipedia(query, max_chars=600):
    """Search Wikipedia and return a summary snippet as a safety fallback."""
    clean = re.sub(r'[^\w\s]', '', query)[:60]
    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{clean.replace(' ', '_')}"
    try:
        r = requests.get(url, timeout=4)
        if r.status_code == 200:
            extract = r.json().get("extract", "")
            if extract:
                return extract[:max_chars]
    except Exception:
        pass

    # Secondary deep title search fallback
    try:
        params = {
            "action": "query", "list": "search",
            "srsearch": query, "format": "json", "srlimit": 1
        }
        r = requests.get("https://en.wikipedia.org/w/api.php", params=params, timeout=4)
        results = r.json().get("query", {}).get("search", [])
        if results:
            title = results[0]["title"]
            return search_wikipedia(title, max_chars)
    except Exception:
        pass

    return ""

def extract_key_terms_with_options(question_text, options):
    """
    Combines question entities with candidate choices.
    This guarantees that the search checks for the tracks/choices explicitly.
    """
    # Isolate any quoted strings first (e.g. "Born to Die")
    quoted = re.findall(r'"([^"]*)"', question_text)
    options_str = " ".join([o.text for o in options])

    # Strip basic filler text
    clean_q = re.sub(r'(Which of these|is not|featured on|the standard version of|the following|correct answer)', '', question_text, flags=re.IGNORECASE)

    if quoted:
        return f'"{quoted[0]}" {options_str}'[:90]
    return f"{clean_q.strip()} {options_str}"[:90]

def build_hybrid_rag_prompt(question_text, options, context=""):
    """
    A smart prompt directing process of elimination when a context string is found.
    """
    opts = "\n".join([f"{chr(65+i)}) {o.text}" for i, o in enumerate(options)])
    ctx_block = f"Background Context Document:\n{context}\n\n" if context else ""
    return (
        f"{ctx_block}"
        f"Task: Answer the multiple-choice trivia question based on the background context provided above.\n"
        f"Rule: If the question contains words like 'NOT', 'EXCEPT', or 'FALSE', eliminate options matched by the context and pick the outlier.\n\n"
        f"Question: {question_text}\n"
        f"Options:\n{opts}\n\n"
        f"Final Answer (Letter Only):"
    )

def answer_with_rag(question):
    """
    Ensemble Retrieval Engine: Queries DuckDuckGo first,
    then falls back to Wikipedia if no result is returned.
    """
    query = extract_key_terms_with_options(question.text, question.options)

    # Step 1: Try Live Web Summary via DuckDuckGo
    context = search_duckduckgo(query)
    source_used = "DuckDuckGo"

    # Step 2: Fall back to Wikipedia if DuckDuckGo came up empty
    if not context:
        context = search_wikipedia(query)
        source_used = "Wikipedia"

    if context:
        print(f"   [RAG Active] Context fetched via {source_used}: '{context[:70]}...'")
    else:
        print("   [RAG Warning] Both sources empty! Flying blind via zero-shot memory.")

    return answer_with_model(
        question,
        lambda q_text, opts: build_hybrid_rag_prompt(q_text, opts, context),
        max_new_tokens=16
    )

print("Hybrid RAG engine successfully compiled! DuckDuckGo + Wikipedia are active.")

In [ ]:
# Run the Multi-Source Hybrid RAG on the Entertainment competition
rag_log, rag_level, rag_earned = play_full_game(
    competition_id=0,
    answer_fn=answer_with_rag,
    label="Qwen2.5 + Hybrid RAG (DDG + Wiki)",
    mode="speech"
)

### 7. Tool Use — Calculator for Maths Category

For the Maths category, we detect mathematical expressions in the question and route them to a Python/SymPy solver rather than relying on the LLM's arithmetic — LLMs are notoriously poor at exact calculation, weak they are.


In [ ]:
# Math solver, implement we shall
import sympy as sp
import re

def extract_and_solve_math(question_text, options):
    """
    Try to solve the question mathematically using SymPy.
    A string with the answer, return we shall — or None if unsolvable.
    """
    # Look for numbers and operators in the question, we must
    # Evaluate simple arithmetic expressions, we can
    try:
        # Find expressions like "12 + 34", "5 * 6", "100 / 4", etc.
        expr_match = re.search(r'[\d]+\s*[\+\-\*\/\^\%]\s*[\d]+', question_text)
        if expr_match:
            expr_str = expr_match.group().replace('^', '**')
            result = sp.sympify(expr_str)
            return str(result)
    except Exception:
        pass

    # Percentage problems, handle we can
    pct_match = re.search(r'(\d+(?:\.\d+)?)%\s+of\s+(\d+(?:\.\d+)?)', question_text, re.IGNORECASE)
    if pct_match:
        pct = float(pct_match.group(1))
        total = float(pct_match.group(2))
        return str(pct * total / 100)

    return None  # Cannot solve symbolically, this problem we cannot

def answer_maths(question):
    # Try math solver first, we shall — fall back to LLM if needed
    solved = extract_and_solve_math(question.text, question.options)

    if solved is not None:
        print(f"   [MATH] Computed answer: {solved}")
        # Match computed answer to closest option, we must
        for i, opt in enumerate(question.options):
            opt_nums = re.findall(r'[\d\.]+', opt.text)
            if opt_nums and abs(float(opt_nums[0]) - float(solved)) < 0.01:
                letter = chr(65 + i)
                print(f"   [MATH] Matched option {letter}: {opt.text}")
                return opt.id, letter, 0.01, f"SymPy: {solved}"
        print("   [MATH] No exact match found, falling back to LLM")

    # Fallback: use the LLM with a math-specific prompt
    return answer_with_model(question, build_zero_shot_prompt)

print("Math tool ready.")

In [ ]:
# Run maths game
maths_log, maths_level, maths_earned = play_full_game(
    competition_id=3,  # Maths
    answer_fn=answer_maths,
    label=" Model + SymPy Tool",
    mode="speech"
)

### 8. Multi-Model Ensemble — Majority Vote

When uncertain, we query multiple models and take a majority vote. When models disagree, choose randomly among the majority.


In [ ]:
# Ensemble: majority vote across models
def answer_ensemble(question):
    votes = {}
    details = []

    models_to_use = [
        ("ZeroShot", lambda q: answer_with_model(q, build_zero_shot_prompt)),
        ("FewShot",  lambda q: answer_with_model(q, build_few_shot_prompt)),
        ("CoT",      lambda q: answer_with_model(q, build_cot_prompt)),
    ]

    for name, fn in models_to_use:
        try:
            opt_id, letter, elapsed, raw = fn(question)
            votes[letter] = votes.get(letter, 0) + 1
            details.append((name, letter, opt_id, elapsed))
            print(f"   [{name}] voted: {letter}")
        except Exception as e:
            print(f"   [{name}] failed: {e}")

    if not votes:
        # All models failed, we choose random answer
        chosen = random.choice(question.options)
        return chosen.id, "?", 0.0, "All models failed"

    # Find most voted letter
    best_letter = max(votes, key=votes.get)
    print(f"   [ENSEMBLE] Majority vote → {best_letter} ({votes[best_letter]}/{len(models_to_use)} votes)")

    # Find option ID for the winning letter
    idx = min(ord(best_letter) - ord("A"), len(question.options) - 1)
    chosen_id = question.options[idx].id
    avg_elapsed = sum(d[3] for d in details) / len(details)

    return chosen_id, best_letter, avg_elapsed, str(votes)

print("Ensemble function ready")

In [ ]:
# Run ensemble game, we shall
ensemble_log, ensemble_level, ensemble_earned = play_full_game(
    competition_id=4,
    answer_fn=answer_ensemble,
    label="Multi-Model Ensemble",
    mode="speech"
)

## 9. Evaluation & Analysis



In [ ]:
# Compile all results into a DataFrame
def log_to_df(log, model_name, competition_id):
    rows = []
    for i, entry in enumerate(log):
        rows.append({
            "model": model_name,
            "competition_id": competition_id,
            "level": entry.get("level", i+1),
            "correct": entry.get("correct", False),
            "timed_out": entry.get("timed_out", False),
            "elapsed_s": entry.get("elapsed", 0),
            "earned": entry.get("earned", 0),
            "question": entry.get("question", ""),
        })
    return pd.DataFrame(rows)

# Combine all logs
all_dfs = []
for log, model_name, comp_id in [
    (baseline_log, "Zero-Shot", COMP_ID),
    (fewshot_log,  "Few-Shot",  COMP_ID),
    (rag_log,      "RAG",       1),
    (maths_log,    "Model + SymPy",   3),
    (ensemble_log, "Ensemble",           COMP_ID),
]:
    if log:
        all_dfs.append(log_to_df(log, model_name, comp_id))

df = pd.concat(all_dfs, ignore_index=True)
print(f"Total game entries logged: {len(df)}")
df.head(10)

In [ ]:
# Summary statistics by model, compute we shall
summary = df.groupby("model").agg(
    questions_answered=("correct", "count"),
    accuracy=("correct", lambda x: x.dropna().mean()),
    avg_response_time=("elapsed_s", "mean"),
    timeouts=("timed_out", "sum"),
    max_earned=("earned", "max"),
).round(3)

print("=== Model Performance Summary ===")
print(summary.to_string())

In [ ]:
# Plot: Accuracy by Model
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Accuracy bar chart
models = summary.index.tolist()
accs = summary["accuracy"].tolist()
axes[0].bar(models, accs, color=["#4CAF50","#2196F3","#FF9800","#9C27B0","#F44336"][:len(models)])
axes[0].set_title("Accuracy by Model")
axes[0].set_ylabel("Accuracy")
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis="x", rotation=30)

# Response time
times = summary["avg_response_time"].tolist()
axes[1].bar(models, times, color="#607D8B")
axes[1].set_title("Avg Response Time (s)")
axes[1].set_ylabel("Seconds")
axes[1].tick_params(axis="x", rotation=30)

# Earnings
earned = summary["max_earned"].tolist()
axes[2].bar(models, earned, color="#FF5722")
axes[2].set_title("Max Earned ($)")
axes[2].set_ylabel("USD")
axes[2].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig("model_comparison.png", dpi=150)
plt.show()
print("Plot saved, it has been.")

In [ ]:
# Accuracy by difficulty level — harder questions, worse models do?
level_acc = df.groupby("level")["correct"].mean().reset_index()
level_acc.columns = ["Level", "Accuracy"]

plt.figure(figsize=(10, 4))
plt.plot(level_acc["Level"], level_acc["Accuracy"], marker="o", color="#2196F3", linewidth=2)
plt.axhline(0.25, linestyle="--", color="red", label="Random chance (25%)")
plt.fill_between(level_acc["Level"], level_acc["Accuracy"], 0.25,
                 where=level_acc["Accuracy"] > 0.25, alpha=0.2, color="green", label="Above chance")
plt.title("Accuracy by Question Level (All Models)")
plt.xlabel("Level")
plt.ylabel("Accuracy")
plt.legend()
plt.tight_layout()
plt.savefig("accuracy_by_level.png", dpi=150)
plt.show()

In [ ]:
# Response time distribution — within 30s, we must stay!
plt.figure(figsize=(10, 4))
for model_name, grp in df.groupby("model"):
    plt.hist(grp["elapsed_s"], bins=15, alpha=0.5, label=model_name)
plt.axvline(30, color="red", linestyle="--", label="30s timeout")
plt.title("Response Time Distribution by Model")
plt.xlabel("Seconds")
plt.ylabel("Count")
plt.legend()
plt.tight_layout()
plt.savefig("response_times.png", dpi=150)
plt.show()

too_slow = df[df["elapsed_s"] > 25]
print(f"Responses dangerously close to timeout (>25s): {len(too_slow)}")

## 11. Research Questions Analysis

The assignment asks us to investigate specific questions. Here we answer them systematically.


In [ ]:
# Q1: Are some models better at certain topics than others?
# Explore, we must — competition_id = category proxy
cat_map = {0: "Entertainment", 1: "Ancient History", 2: "Science", 3: "Maths"}

cat_acc = df.groupby(["model","competition_id"])["correct"].mean().reset_index()
cat_acc["category"] = cat_acc["competition_id"].map(cat_map)

pivot = cat_acc.pivot(index="model", columns="category", values="correct")
print("=== Accuracy by Model and Category ===")
print(pivot.round(2).to_string())

In [ ]:
# Q2: Is the model overconfident? (can check via prompt output entropy — rough approximation)
# Q3: What types of questions do models struggle on?
wrong_qs = df[df["correct"] == False][["model","level","question"]].dropna()
print(f"=== Sample of Wrong Answers ({len(wrong_qs)} total) ===")
print(wrong_qs.head(10).to_string(index=False))

In [ ]:
# Q4: Does RAG help vs. not?
# Compare Flan-T5 Zero-Shot vs Flan-T5 RAG on the same competition (if available)
rag_comp = df[df["model"].isin(["Flan-T5 Zero-Shot", "Flan-T5 RAG"])]
if len(rag_comp) > 0:
    comparison = rag_comp.groupby("model")["correct"].mean()
    print("=== RAG vs. No RAG ===")
    print(comparison)
    improvement = comparison.get("Flan-T5 RAG", 0) - comparison.get("Flan-T5 Zero-Shot", 0)
    print(f"RAG improvement: {improvement:+.1%}")

## 12. Best System — Final Run

Based on our evaluation above, the best configuration select we shall and run one clean game for the leaderboard.


In [ ]:
# Best strategy, define we must based on evaluation results above
# Update this after running all experiments!

def best_strategy(question):
    """
    Winning strategy, this is. Combine RAG + few-shot + math tool, we do.
    - Maths questions → SymPy tool
    - Other questions → RAG + few-shot prompt
    """
    question_text_lower = question.text.lower()

    # Math detection heuristic
    math_keywords = ["calculate","compute","solve","equation","percentage","%" ,"sum",
                     "product","divided","multiplied","squared","factorial","derivative"]
    is_math = any(kw in question_text_lower for kw in math_keywords)

    if is_math:
        return answer_maths(question)
    else:
        return answer_with_rag(question)

print("Best strategy ready, it is. To the leaderboard, go we shall!")

In [ ]:
# Play all 4 competitions with the best strategy, we shall
comp_names = {0: "Entertainment", 1: "Ancient History", 2: "Science & Nature", 3: "Maths"}
final_results = {}

for comp_id in [0, 1, 2, 3]:
    print(f"\n{'='*60}")
    print(f"Playing: {comp_names[comp_id]}")
    print(f"{'='*60}")
    log, level, earned = play_full_game(
        competition_id=comp_id,
        answer_fn=best_strategy,
        label=f"Best System — {comp_names[comp_id]}"
    )
    final_results[comp_id] = {"level": level, "earned": earned}
    time.sleep(2)  # Be polite to the server, we must

print("\n=== FINAL RESULTS ===")
for cid, res in final_results.items():
    print(f"  {comp_names[cid]}: Level {res['level']} | ${res['earned']:,.0f}")

In [ ]:
# Check leaderboard positions, we shall
print("=== Leaderboard Positions ===")
for comp_id in [0, 1, 2, 3]:
    lb = client.leaderboard.get(competition_id=comp_id, limit=20)
    print(f"\n--- {lb.competition.name} ---")
    for i, entry in enumerate(lb.entries[:10], 1):
        marker = " ← YOU" if entry.username == USERNAME else ""
        print(f"  {i}. {entry.username}: ${entry.score:,.0f} (Level {entry.reached_level}){marker}")

## 13. Conclusions

Summarize our findings here, we must.

### Key Findings

| Question | Finding |
|----------|---------|
| Zero-shot vs few-shot vs CoT? | *(fill in after experiments)* |
| Does RAG improve accuracy? | *(fill in after experiments)* |
| Does SymPy help for Maths? | *(fill in after experiments)* |
| Are bigger models better? | *(fill in after experiments)* |
| Can we answer within 30s? | *(fill in after experiments)* |
| Which category is hardest? | *(fill in after experiments)* |

### Future Work
- Fine-tuning on domain-specific data
- Speech interface experiments
- More powerful models (Mistral-7B, Qwen 7B) with 4-bit quantization
- Confidence-based routing between models

*Completed this assignment, we have. May the Force of NLP be with you.*
